In [1]:
!nvidia-smi

Thu Oct  8 07:06:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from numba import cuda
import os

device = cuda.get_current_device()

print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


In [4]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out


# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Calculate unique 1D thread ID
    idx = cuda.grid(1)

    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [5]:
# ============================================================
# TASK 2.2: CPU vs GPU CROSSOVER BENCHMARK
# ============================================================

import numpy as np
import time
from numba import cuda


# ------------------------------------------------------------
# GPU benchmark function
# ------------------------------------------------------------
def benchmark_gpu(N):

    # Create input data on CPU
    h_in = np.ones(N, dtype=np.float32)

    # Allocate GPU memory
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)

    # Thread configuration
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warm-up run
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()

    # --------------------------------------------------------
    # GPU TOTAL TIME
    # Host -> GPU + Kernel + GPU -> Host
    # --------------------------------------------------------
    start_total = time.perf_counter()

    d_in = cuda.to_device(h_in)

    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)

    result = d_out.copy_to_host()

    cuda.synchronize()

    end_total = time.perf_counter()

    gpu_total_ms = (end_total - start_total) * 1000


    # --------------------------------------------------------
    # GPU KERNEL-ONLY TIME
    # Data is already in GPU memory
    # --------------------------------------------------------

    start_kernel = time.perf_counter()

    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)

    cuda.synchronize()

    end_kernel = time.perf_counter()

    gpu_kernel_ms = (end_kernel - start_kernel) * 1000


    # Free GPU memory
    del d_in
    del d_out

    return gpu_total_ms, gpu_kernel_ms


# ------------------------------------------------------------
# CPU benchmark function
# ------------------------------------------------------------
def benchmark_cpu(N):

    h_in = np.ones(N, dtype=np.float32)

    start = time.perf_counter()

    result = double_cpu_pure(h_in)

    end = time.perf_counter()

    cpu_ms = (end - start) * 1000

    return cpu_ms


# ------------------------------------------------------------
# Array sizes required by the laboratory protocol
# ------------------------------------------------------------
sizes = [
    100,
    1_000,
    10_000,
    100_000,
    1_000_000,
    5_000_000,
    10_000_000
]


# ------------------------------------------------------------
# Run experiment
# ------------------------------------------------------------
print("=" * 85)
print("CUDA VECTOR DOUBLING CROSSOVER EXPERIMENT")
print("=" * 85)

print(
    f"{'N':>12} | "
    f"{'CPU (ms)':>12} | "
    f"{'GPU Total (ms)':>16} | "
    f"{'GPU Kernel (ms)':>17} | "
    f"{'Winner':>10}"
)

print("-" * 85)


results = []

for N in sizes:

    cpu_time = benchmark_cpu(N)

    gpu_total, gpu_kernel = benchmark_gpu(N)

    if cpu_time < gpu_total:
        winner = "CPU"
    else:
        winner = "GPU"

    results.append(
        (N, cpu_time, gpu_total, gpu_kernel, winner)
    )

    print(
        f"{N:12,d} | "
        f"{cpu_time:12.4f} | "
        f"{gpu_total:16.4f} | "
        f"{gpu_kernel:17.4f} | "
        f"{winner:>10}"
    )

print("=" * 85)

CUDA VECTOR DOUBLING CROSSOVER EXPERIMENT
           N |     CPU (ms) |   GPU Total (ms) |   GPU Kernel (ms) |     Winner
-------------------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


         100 |       0.0484 |           0.6825 |            0.0684 |        CPU
       1,000 |       0.2554 |           0.4675 |            0.0581 |        CPU
      10,000 |       2.5193 |           0.4463 |            0.0653 |        GPU
     100,000 |      22.9133 |           0.6327 |            0.0792 |        GPU


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


   1,000,000 |     241.9392 |           3.8794 |            0.1060 |        GPU
   5,000,000 |    2155.7146 |          13.3685 |            0.2888 |        GPU
  10,000,000 |    2808.3094 |          23.0780 |            0.4640 |        GPU


In [6]:
import numpy as np
from numba import cuda

# ============================================================
# SECTION 3.1: MISSING GUARD DISASTER
# ============================================================

N = 1000
threads_per_block = 256

# Ceiling division:
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

print("Array Size (N):", N)
print("Threads Per Block:", threads_per_block)
print("Blocks Per Grid:", blocks_per_grid)
print("Total Threads Launched:", blocks_per_grid * threads_per_block)


# ------------------------------------------------------------
# DEFECTIVE KERNEL
# NO BOUNDARY GUARD!
# ------------------------------------------------------------
@cuda.jit
def defective_kernel(d_in, d_out):

    idx = cuda.grid(1)

    # INTENTIONALLY NO:
    # if idx < d_in.shape[0]:

    d_out[idx] = d_in[idx] * 2.0


# ------------------------------------------------------------
# Create input array
# ------------------------------------------------------------
h_in = np.ones(N, dtype=np.float32)

# Allocate GPU memory
d_in = cuda.to_device(h_in)
d_out = cuda.device_array(N, dtype=np.float32)


# ------------------------------------------------------------
# Launch defective kernel
# ------------------------------------------------------------
try:

    defective_kernel[blocks_per_grid, threads_per_block](d_in, d_out)

    cuda.synchronize()

    print("\nKernel completed without a reported error.")

    result = d_out.copy_to_host()

    print("First 10 output values:", result[:10])

except Exception as e:

    print("\nCUDA ERROR DETECTED:")
    print(type(e).__name__)
    print(e)

Array Size (N): 1000
Threads Per Block: 256
Blocks Per Grid: 4
Total Threads Launched: 1024

Kernel completed without a reported error.
First 10 output values: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]


In [7]:
import numpy as np
from numba import cuda

# ============================================================
# SECTION 3.2: SPATIAL THREAD COORDINATE INSPECTOR
# ============================================================

N = 37
threads_per_block = 8

# Ceiling division
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

print("N:", N)
print("Threads Per Block:", threads_per_block)
print("Blocks Per Grid:", blocks_per_grid)
print("Total Threads Launched:", blocks_per_grid * threads_per_block)


# ------------------------------------------------------------
# Diagnostic kernel
# ------------------------------------------------------------
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):

    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:

        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx

        if g_idx < N:
            log_active[g_idx] = 1
        else:
            log_active[g_idx] = 0


# ------------------------------------------------------------
# Allocate logging arrays
# ------------------------------------------------------------

total_threads = blocks_per_grid * threads_per_block

log_b = cuda.device_array(total_threads, dtype=np.int32)
log_t = cuda.device_array(total_threads, dtype=np.int32)
log_g = cuda.device_array(total_threads, dtype=np.int32)
log_a = cuda.device_array(total_threads, dtype=np.int32)


# ------------------------------------------------------------
# Launch kernel
# ------------------------------------------------------------

inspect_threads_kernel[
    blocks_per_grid,
    threads_per_block
](
    log_b,
    log_t,
    log_g,
    log_a,
    N
)

cuda.synchronize()


# ------------------------------------------------------------
# Copy results back to CPU
# ------------------------------------------------------------

log_b_host = log_b.copy_to_host()
log_t_host = log_t.copy_to_host()
log_g_host = log_g.copy_to_host()
log_a_host = log_a.copy_to_host()


# ------------------------------------------------------------
# Display boundary transition
# Global IDs 32-39 are the final block
# ------------------------------------------------------------

print()
print(f"{'Global ID':<12} | {'Block ID':<10} | {'Thread ID':<11} | {'Status':<18}")
print("-" * 60)

for i in range(32, 40):

    if log_a_host[i] == 1:
        status = "ACTIVE"
    else:
        status = "IDLE (GUARDED)"

    print(
        f"{log_g_host[i]:<12} | "
        f"{log_b_host[i]:<10} | "
        f"{log_t_host[i]:<11} | "
        f"{status:<18}"
    )


# ------------------------------------------------------------
# Mandatory calculations
# ------------------------------------------------------------

active_threads = N
idle_threads = total_threads - N

print()
print("MANDATORY CALCULATION")
print("---------------------")
print("Total Blocks Allocated:", blocks_per_grid)
print("Total Hardware Threads Launched:", total_threads)
print("Number of Active Working Threads:", active_threads)
print("Number of Idle Guarded Threads:", idle_threads)

N: 37
Threads Per Block: 8
Blocks Per Grid: 5
Total Threads Launched: 40

Global ID    | Block ID   | Thread ID   | Status            
------------------------------------------------------------
32           | 4          | 0           | ACTIVE            
33           | 4          | 1           | ACTIVE            
34           | 4          | 2           | ACTIVE            
35           | 4          | 3           | ACTIVE            
36           | 4          | 4           | ACTIVE            
37           | 4          | 5           | IDLE (GUARDED)    
38           | 4          | 6           | IDLE (GUARDED)    
39           | 4          | 7           | IDLE (GUARDED)    

MANDATORY CALCULATION
---------------------
Total Blocks Allocated: 5
Total Hardware Threads Launched: 40
Number of Active Working Threads: 37
Number of Idle Guarded Threads: 3


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [8]:
import numpy as np
import time
from numba import cuda


# ============================================================
# SECTION 4: RACE CONDITION CRIME SCENE
# ============================================================


# ------------------------------------------------------------
# NAIVE REDUCTION KERNEL
# Intentionally contains a race condition
# ------------------------------------------------------------
@cuda.jit
def naive_sum_kernel(arr, total):

    idx = cuda.grid(1)

    if idx < arr.size:

        # RACE CONDITION:
        # Many threads read and write total[0]
        # simultaneously.
        total[0] += arr[idx]


# ------------------------------------------------------------
# ATOMIC REDUCTION KERNEL
# Correct version using atomic addition
# ------------------------------------------------------------
@cuda.jit
def atomic_sum_kernel(arr, total):

    idx = cuda.grid(1)

    if idx < arr.size:

        cuda.atomic.add(total, 0, arr[idx])


# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

N = 50_000

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

print("=" * 70)
print("SECTION 4: RACE CONDITION EXPERIMENT")
print("=" * 70)

print("Array Size:", N)
print("Threads Per Block:", threads_per_block)
print("Blocks Per Grid:", blocks_per_grid)
print("Expected Sum:", float(N))


# ------------------------------------------------------------
# Create array of 50,000 ones
# ------------------------------------------------------------

h_arr = np.ones(N, dtype=np.float32)

d_arr = cuda.to_device(h_arr)


# ============================================================
# TASK 4.1 — NAIVE KERNEL: FIVE TRIALS
# ============================================================

print()
print("NAIVE KERNEL — 5 CONSECUTIVE TRIALS")
print("-" * 70)

naive_results = []

for trial in range(1, 6):

    # Reset total to zero
    d_total = cuda.to_device(
        np.array([0.0], dtype=np.float32)
    )

    # Run kernel
    naive_sum_kernel[
        blocks_per_grid,
        threads_per_block
    ](
        d_arr,
        d_total
    )

    cuda.synchronize()

    # Copy result back
    result = d_total.copy_to_host()[0]

    deviation = float(N) - float(result)

    naive_results.append((result, deviation))

    print(
        f"Trial {trial}: "
        f"Target = {N:.1f}, "
        f"Output = {result:.1f}, "
        f"Deviation / Lost Updates = {deviation:.1f}"
    )


# ============================================================
# ATOMIC KERNEL
# ============================================================

print()
print("ATOMIC KERNEL")
print("-" * 70)

d_total_atomic = cuda.to_device(
    np.array([0.0], dtype=np.float32)
)

atomic_sum_kernel[
    blocks_per_grid,
    threads_per_block
](
    d_arr,
    d_total_atomic
)

cuda.synchronize()

atomic_result = d_total_atomic.copy_to_host()[0]

print("Atomic Sum Output:", atomic_result)
print("Expected Sum:", float(N))
print("Correct:", atomic_result == float(N))


# ============================================================
# PERFORMANCE COMPARISON
# ============================================================

# ----------------------------
# Time naive kernel
# ----------------------------

d_total_naive = cuda.to_device(
    np.array([0.0], dtype=np.float32)
)

# Warm-up
naive_sum_kernel[
    blocks_per_grid,
    threads_per_block
](d_arr, d_total_naive)

cuda.synchronize()


start = time.perf_counter()

naive_sum_kernel[
    blocks_per_grid,
    threads_per_block
](d_arr, d_total_naive)

cuda.synchronize()

end = time.perf_counter()

naive_time_ms = (end - start) * 1000


# ----------------------------
# Time atomic kernel
# ----------------------------

d_total_atomic = cuda.to_device(
    np.array([0.0], dtype=np.float32)
)

# Warm-up
atomic_sum_kernel[
    blocks_per_grid,
    threads_per_block
](d_arr, d_total_atomic)

cuda.synchronize()


start = time.perf_counter()

atomic_sum_kernel[
    blocks_per_grid,
    threads_per_block
](d_arr, d_total_atomic)

cuda.synchronize()

end = time.perf_counter()

atomic_time_ms = (end - start) * 1000


# ============================================================
# FINAL PERFORMANCE OUTPUT
# ============================================================

print()
print("=" * 70)
print("PERFORMANCE COMPARISON")
print("=" * 70)

print(f"Naive Kernel Time:  {naive_time_ms:.4f} ms")
print(f"Atomic Kernel Time: {atomic_time_ms:.4f} ms")

print()
print("Race Condition Demonstrated: Multiple threads")
print("attempted to update the same memory location.")
print("Atomic operations serialize conflicting updates")
print("to guarantee correctness.")

SECTION 4: RACE CONDITION EXPERIMENT
Array Size: 50000
Threads Per Block: 256
Blocks Per Grid: 196
Expected Sum: 50000.0

NAIVE KERNEL — 5 CONSECUTIVE TRIALS
----------------------------------------------------------------------
Trial 1: Target = 50000.0, Output = 2.0, Deviation / Lost Updates = 49998.0
Trial 2: Target = 50000.0, Output = 2.0, Deviation / Lost Updates = 49998.0
Trial 3: Target = 50000.0, Output = 2.0, Deviation / Lost Updates = 49998.0
Trial 4: Target = 50000.0, Output = 2.0, Deviation / Lost Updates = 49998.0
Trial 5: Target = 50000.0, Output = 2.0, Deviation / Lost Updates = 49998.0

ATOMIC KERNEL
----------------------------------------------------------------------
Atomic Sum Output: 50000.0
Expected Sum: 50000.0
Correct: True

PERFORMANCE COMPARISON
Naive Kernel Time:  0.1803 ms
Atomic Kernel Time: 0.3120 ms

Race Condition Demonstrated: Multiple threads
attempted to update the same memory location.
Atomic operations serialize conflicting updates
to guarantee corr

In [9]:
import numpy as np
from numba import cuda


# ============================================================
# SECTION 5: 2D GRID IMAGE / MATRIX PROCESSING
# ============================================================

WIDTH = 1024
HEIGHT = 1024

threads_x = 16
threads_y = 16


# ------------------------------------------------------------
# Calculate grid dimensions
# ------------------------------------------------------------

blocks_x = (WIDTH + threads_x - 1) // threads_x
blocks_y = (HEIGHT + threads_y - 1) // threads_y

total_blocks = blocks_x * blocks_y


print("=" * 70)
print("SECTION 5: 2D RADIAL VIGNETTE")
print("=" * 70)

print("Image Width:", WIDTH)
print("Image Height:", HEIGHT)
print("Threads Per Block:", threads_x, "x", threads_y)
print("Blocks in X:", blocks_x)
print("Blocks in Y:", blocks_y)
print("Total Blocks:", total_blocks)


# ------------------------------------------------------------
# CUDA 2D kernel
# ------------------------------------------------------------

@cuda.jit
def radial_vignette_kernel(output):

    x, y = cuda.grid(2)

    height = output.shape[0]
    width = output.shape[1]

    if x < width and y < height:

        # Center of image
        cx = width / 2.0
        cy = height / 2.0

        # Distance from center
        dx = x - cx
        dy = y - cy

        distance = (dx * dx + dy * dy) ** 0.5

        # Maximum distance to corner
        max_distance = (cx * cx + cy * cy) ** 0.5

        # Normalize distance
        normalized_distance = distance / max_distance

        # Vignette intensity
        value = 1.0 - normalized_distance

        # Clamp value
        if value < 0.0:
            value = 0.0

        output[y, x] = value


# ------------------------------------------------------------
# Allocate GPU output
# ------------------------------------------------------------

d_output = cuda.device_array(
    (HEIGHT, WIDTH),
    dtype=np.float32
)


# ------------------------------------------------------------
# Launch 2D CUDA kernel
# ------------------------------------------------------------

radial_vignette_kernel[
    (blocks_x, blocks_y),
    (threads_x, threads_y)
](d_output)

cuda.synchronize()


# ------------------------------------------------------------
# Copy result back to CPU
# ------------------------------------------------------------

output = d_output.copy_to_host()


# ------------------------------------------------------------
# Verify required pixels
# ------------------------------------------------------------

center_value = output[512, 512]
corner_value = output[0, 0]


print()
print("MANDATORY 2D GEOMETRY VERIFICATION")
print("-" * 70)

print(
    "Grid Dimensions:",
    f"Blocks in X: {blocks_x} |",
    f"Blocks in Y: {blocks_y} |",
    f"Total Blocks: {total_blocks}"
)

print(
    "Verified Pixel Value at Center Point (512, 512):",
    center_value
)

print(
    "Verified Pixel Value at Outer Corner Point (0, 0):",
    corner_value
)

SECTION 5: 2D RADIAL VIGNETTE
Image Width: 1024
Image Height: 1024
Threads Per Block: 16 x 16
Blocks in X: 64
Blocks in Y: 64
Total Blocks: 4096

MANDATORY 2D GEOMETRY VERIFICATION
----------------------------------------------------------------------
Grid Dimensions: Blocks in X: 64 | Blocks in Y: 64 | Total Blocks: 4096
Verified Pixel Value at Center Point (512, 512): 1.0
Verified Pixel Value at Outer Corner Point (0, 0): 0.0


In [11]:
import hashlib
from numba import cuda


def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):

    h = hashlib.sha256()

    # --------------------------------------------------------
    # Include Student ID
    # --------------------------------------------------------
    h.update(
        student_id_str.strip().encode('utf-8')
    )

    # --------------------------------------------------------
    # Include GPU name
    # FIX: encode GPU name before hashing
    # --------------------------------------------------------
    gpu_name = cuda.get_current_device().name

    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode('utf-8')

    h.update(gpu_name)

    # --------------------------------------------------------
    # Include sample of compute output
    # --------------------------------------------------------
    h.update(
        d_canvas.copy_to_host()[:10, :10].tobytes()
    )

    # --------------------------------------------------------
    # Include crossover experiment result
    # --------------------------------------------------------
    h.update(
        str(exp_table_crossover).encode('utf-8')
    )

    # --------------------------------------------------------
    # Generate 16-character token
    # --------------------------------------------------------
    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest


# ============================================================
# YOUR LAB INFORMATION
# ============================================================

STUDENT_ID = "230103057"

# First tested N where GPU Total Time became faster
CROSSOVER_N = 10000


# ============================================================
# GENERATE CHECKSUM
# ============================================================

generate_lab_checksum(
    STUDENT_ID,
    d_output,
    CROSSOVER_N
)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 2753A1CC596BD9E0


'2753A1CC596BD9E0'